<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Similarity and distance

**[Similarity and distance](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/)** · Essential Mathematics and Statistics for AI · Module 2, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** rank four solved tickets by cosine similarity and by Euclidean distance, see why the two rankings disagree, standardise two features in different units, and do the module practice with three features.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The dot product and the length of a vector, from [Vector operations: The dot product](https://learning.nextia-ai.com/courses/math/m02/vector-operations/#the-dot-product-a-weighted-score) and [The length of a vector](https://learning.nextia-ai.com/courses/math/m02/vector-operations/#the-length-of-a-vector). The mean and the SD, from [Describe a dataset: Spread](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#spread-range-variance-and-standard-deviation). |
| **Tested** | Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M02-L03-similarity-and-distance/similarity-and-distance-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two ways to say "close"

The new ticket **T-4200** says: "I want a refund, my parcel is late." Priya counts refund words and delivery words: T-4200 = (2, 1).

The next cell defines the tools from Lesson 2, `dot` and `length`, and two new ones:

- **Euclidean distance**: subtract position by position, square, add, and take the square root. It is the length of the difference vector.
- **Cosine similarity**: the dot product divided by both lengths. Only the direction is left.

> **Predict.** T-3912 = (1, 2). What is its distance from T-4200? Then run the cell.

In [ ]:
def dot(u, v):
    """Multiply position by position, then add the products."""
    return sum(a * b for a, b in zip(u, v))

def length(v):
    """The length of v: the square root of v · v."""
    return math.sqrt(dot(v, v))

def distance(u, v):
    """Euclidean distance: the length of the difference vector u − v."""
    return length([a - b for a, b in zip(u, v)])

def cosine(u, v):
    """Cosine similarity: the dot product divided by both lengths."""
    return dot(u, v) / (length(u) * length(v))

t4200 = [2, 1]
t3912 = [1, 2]
print("difference          ", [a - b for a, b in zip(t4200, t3912)])
print("Euclidean distance  ", round(distance(t4200, t3912), 3))
print("cosine similarity   ", round(cosine(t4200, t3912), 3))

> **Check.** You should see the difference `[1, -1]`, a distance of `1.414` (√2) and a cosine similarity of `0.8`.

A cosine similarity is a number from −1 to 1. The next cell tries three pairs with a clear direction.

> **Predict.** What is the cosine similarity of (2, 1) with (4, 2), with (−1, 2) and with (−2, −1)? Then run the cell.

In [ ]:
for v in [[4, 2], [-1, 2], [-2, -1]]:
    print(f"(2, 1) and {str(tuple(v)):8}  cosine {cosine(t4200, v):5.2f}")

> **Check.** You should see `1.00` (the same direction), `0.00` (a right angle) and `-1.00` (opposite directions). The lengths do not matter: (4, 2) is twice as long as (2, 1).

### A vector of length 0

A ticket with no refund words and no delivery words is (0, 0). Its length is 0, and the cosine similarity divides by 0.

> **Predict.** What does Python do with `cosine([2, 1], [0, 0])`? Then run the cell.

In [ ]:
try:
    print(cosine(t4200, [0, 0]))
except ZeroDivisionError as error:
    print("ZeroDivisionError:", error)
print("the distance still exists:", round(distance(t4200, [0, 0]), 3))

> **Check.** You should see `ZeroDivisionError: division by zero`. A vector of length 0 has no direction, so the cosine similarity is **not defined**. The distance, 2.236, still exists. A search system must handle this case, for example by skipping empty tickets.

## 2. Worked example: similar past tickets for T-4200

Priya compares T-4200 with four solved tickets. The next cell makes her table. The **angle** comes from the cosine similarity: `math.acos` turns it back into an angle. (`min(1, ...)` stops a tiny rounding error, such as 1.0000000000000002, from breaking `acos`.)

> **Predict.** Which ticket has the highest cosine similarity? Which has the smallest distance? Is it the same ticket? Then run the cell.

In [ ]:
past = {
    "T-3907": [6, 3],   # a long complaint: refund for a late, damaged parcel
    "T-3915": [4, 1],   # a refund for a wrong item, delivered on time
    "T-3912": [1, 2],   # mostly about a late parcel, one mention of a refund
    "T-3920": [0, 4],   # only about tracking a parcel
}

def angle(u, v):
    """The angle between u and v, in degrees."""
    return math.degrees(math.acos(min(1, cosine(u, v))))

rows = []
for ticket, v in past.items():
    rows.append([ticket, str(tuple(v)), dot(t4200, v), length(v), cosine(t4200, v),
                 f"{angle(t4200, v):.1f}°", distance(t4200, v)])
print("length of T-4200:", round(length(t4200), 3))
table(["Ticket", "Vector", "Dot", "Length", "Cosine", "Angle", "Distance"], rows, digits=3)

> **Check.** You should see the cosine similarities 1.000, 0.976, 0.800 and 0.447, the angles 0.0°, 12.5°, 36.9° and 63.4°, and the distances 4.472, 2.000, 1.414 and 3.606. The length of T-4200 is 2.236.

Now sort the tickets both ways. `sorted(..., key=...)` sorts by a number that you give for each ticket.

Run the cell. You should see T-3907 first by cosine similarity, but last by distance.

In [ ]:
by_cosine = sorted(past, key=lambda t: cosine(t4200, past[t]), reverse=True)   # highest first
by_distance = sorted(past, key=lambda t: distance(t4200, past[t]))              # smallest first
table(["Rank", "By cosine similarity", "By Euclidean distance"],
      [[i + 1, c, d] for i, (c, d) in enumerate(zip(by_cosine, by_distance))])

> **Check.** By cosine similarity: T-3907, T-3915, T-3912, T-3920. By distance: T-3912, T-3915, T-3920, T-3907.

Grace says that T-3907 is the same problem as T-4200: the customer only wrote more. **Cosine similarity ignores length**, so it puts T-3907 first. **Euclidean distance includes length**, so the long ticket is far away. On the lesson page, the [explorer](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/#worked-example-similar-past-tickets-for-t-4200) lets you drag the two tips.

## 3. Feature scaling: when one unit decides everything

Priya tests a search with two features: length and refund words. N is a new refund request. P is a shorter refund request: the same problem. Q is a question about a garden chair, with no refund words.

First, length is in hundreds of words. Then Mei's pipeline counts length in words.

> **Predict.** In which unit is P closer to N? In which unit is Q closer? Then run the cell.

In [ ]:
in_hundreds = {"N": [2, 3], "P": [1, 3], "Q": [2.2, 0]}
in_words = {"N": [200, 3], "P": [100, 3], "Q": [220, 0]}

for unit, tickets in [("hundreds of words", in_hundreds), ("words", in_words)]:
    n = tickets["N"]
    print(f"Length in {unit}:")
    print(f"  d(N, P) = {distance(n, tickets['P']):7.3f}   cosine {cosine(n, tickets['P']):.5f}")
    print(f"  d(N, Q) = {distance(n, tickets['Q']):7.3f}   cosine {cosine(n, tickets['Q']):.5f}")

> **Check.** In hundreds of words, d(N, P) = 1.000 and d(N, Q) = 3.007: P is closer, which is correct. In words, d(N, P) = 100.000 and d(N, Q) = 20.224: now Q is closer. In words, the cosine similarity is 0.99989 for both, so it cannot tell P and Q apart. (In hundreds of words, it could: 0.96476 and 0.55470.)

Distance adds the squared differences of all features. The next cell shows the share of each feature in the squared distance, for the explorer's two tickets a = (2, 3) and b = (3, 1).

> **Predict.** Length gives 20% of the squared distance. What share does it give when you count length in words (× 100)? Then run the cell.

In [ ]:
def shares(u, v):
    """The share of each feature in the squared distance, in percent."""
    squares = [(a - b) ** 2 for a, b in zip(u, v)]
    return [round(s / sum(squares) * 100, 2) for s in squares]

for label, a, b in [("hundreds of words", [2, 3], [3, 1]), ("words (× 100)", [200, 3], [300, 1])]:
    print(f"Length in {label}:")
    bars(["length", "refund words"], shares(a, b))

> **Check.** You should see 20.0 and 80.0 at first. In words, length gives 99.96% of the squared distance, and refund words only 0.04%. The bar for refund words is empty.

### The fix: standardise each feature

For each feature, subtract its mean and divide by its standard deviation (SD), both from past tickets. On last month's tickets, length has mean 200 words and SD 100 words. Refund words have mean 1.5 and SD 1.5.

Run the cell. You should see N `[0.0, 1.0]`, P `[-1.0, 1.0]` and Q `[0.2, -1.0]`, then `d(N, P) = 1.0` and `d(N, Q) = 2.01`. P is closer again.

In [ ]:
def standardise(value, mean, sd):
    """How many SDs the value is above (+) or below (−) the mean."""
    return (value - mean) / sd

z = {t: [standardise(v[0], 200, 100), standardise(v[1], 1.5, 1.5)] for t, v in in_words.items()}
for t, v in z.items():
    print(t, v)
print("d(N, P) =", round(distance(z["N"], z["P"]), 3))
print("d(N, Q) =", round(distance(z["N"], z["Q"]), 3))

> **Your turn: change one thing.** In the cell above, use `in_hundreds` instead of `in_words`, and the mean 2 and SD 1 for length instead of 200 and 100. Run it again. You should see the same z values: after standardising, the unit no longer matters. Python can print 0.2 as `0.20000000000000018`: that is the same number, with a tiny rounding error.

In scikit-learn, this step is `StandardScaler`. It uses the mean and the SD of the past tickets for every new ticket.

## 4. A similarity score is not a probability

A cosine similarity of 0.93 does not mean "93% sure". It measures an angle. Read the three cautions in [How a similarity score misleads](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/#how-a-similarity-score-misleads) on the lesson page. The only way to know what a score means is to test it on pairs that a person labelled.

## 5. Guided practice: rank one more ticket

One more solved ticket, **T-3925**, has the vector (3, 4): 3 refund words, 4 delivery words.

> **Your turn.** Replace each `...`. Calculate each value step by step, with numbers or with the functions above. Then give the place of T-3925 in each ranking of section 2, when you add it to the four tickets (1 is the most similar). Run the check cell.

In [ ]:
t3925 = [3, 4]

dot_3925 = dot(t4200, t3925)
length_3925 = length(t3925)
cosine_3925 = dot_3925 / (length(t4200) * length_3925)
difference = [a - b for a, b in zip(t4200, t3925)]
distance_3925 = length(difference)
five = {**past, "T-3925": t3925}
place_cosine = sorted(five, key=lambda t: cosine(t4200, five[t]), reverse=True).index("T-3925") + 1
place_distance = sorted(five, key=lambda t: distance(t4200, five[t])).index("T-3925") + 1

Run the check cell.

In [ ]:
expect("the dot product", dot_3925, 10)
expect("the length of T-3925", length_3925, 5)
expect("the cosine similarity", cosine_3925, 0.894, tol=0.001)
expect("the difference vector", difference, [-1, -3])
expect("the Euclidean distance", distance_3925, 3.162, tol=0.001)
expect("the place by cosine similarity", place_cosine, 3)
expect("the place by distance", place_distance, 3)

A cosine similarity is never above 1 and never below −1. If yours is outside that range, check the lengths.

## 6. Independent variation: scale every vector to length 1

Priya first divides every vector by its own length, so all vectors have length 1.

> **Predict.** Which is closer to T-4200 now, T-3907 or T-3912?

> **Your turn.** Write a function `unit(v)` that divides each number of v by the length of v. Then calculate the distance between the unit vectors of T-4200 and T-3907, and between the unit vectors of T-4200 and T-3912. Run the check cell.

In [ ]:
def unit(v):
    return [x / length(v) for x in v]

d_3907 = distance(unit(t4200), unit(past["T-3907"]))
d_3912 = distance(unit(t4200), unit(past["T-3912"]))

Run the check cell. It also checks your `unit` function on T-4200.

In [ ]:
first = unit(t4200)   # None while the function body is still ...
expect("the first number of unit(T-4200)", ... if first is None else first[0], 0.894, tol=0.001)
expect("the distance to T-3907", d_3907, 0, tol=0.001)
expect("the distance to T-3912", d_3912, 0.632, tol=0.001)

T-3907 is now closer, as in the cosine ranking. For unit vectors, the squared distance is 2 − 2 × cosine similarity. Run the cell to see it for all four tickets. You should see two equal columns: 0.000, 0.048, 0.400 and 1.106.

In [ ]:
table(["Ticket", "Squared distance of unit vectors", "2 − 2 × cosine"],
      [[t, distance([x / length(t4200) for x in t4200], [x / length(v) for x in v]) ** 2,
        2 - 2 * cosine(t4200, v)] for t, v in past.items()], digits=3)

## 7. Failure case: one feature in large units decides the search

After a pipeline change, the similar-ticket search shows tickets of the same length but a different topic. The fix in the lesson: standardise, then test the search with pairs that you know. Here the known pair is N and P: P is the same problem, so it must be closer to N than Q is.

Run the cell. You should see `FAILED` for the raw lengths in words, and `passed` after standardising.

In [ ]:
def known_pair_test(tickets):
    """The known duplicate P must be closer to N than the unrelated Q."""
    ok = distance(tickets["N"], tickets["P"]) < distance(tickets["N"], tickets["Q"])
    return "passed" if ok else "FAILED"

print("length in words, raw:   ", known_pair_test(in_words))
print("length in words, z:     ", known_pair_test(z))

**Fix:** standardise every feature with the mean and the SD from past tickets before you measure distance. Run a test like this again after every change to the pipeline.

## 8. Module practice: three features, by hand

### Worked example

A new ticket, **T-4300**, has three counts: refund words, delivery words and account words. **u** = (2, 1, 2). Priya compares it with **P1** = (1, 2, 2). Run the cell. You should see the products 2, 2 and 4, the dot product 8, both lengths 3.000, a cosine similarity of 0.889 and a distance of 1.414.

In [ ]:
u = [2, 1, 2]
p1 = [1, 2, 2]

table(["Step", "Result"], [
    ["Products", str([a * b for a, b in zip(u, p1)])],
    ["Dot product u · P1", dot(u, p1)],
    ["Length of u", length(u)],
    ["Length of P1", length(p1)],
    ["Cosine similarity", cosine(u, p1)],
    ["Difference u − P1", str([a - b for a, b in zip(u, p1)])],
    ["Euclidean distance", distance(u, p1)],
], digits=3)

Both measures say "similar". The cosine similarity of 0.889 is not an 89% chance of anything.

### Your task

Two more solved tickets: **P2** = (4, 0, 3) and **P3** = (0, 3, 4).

> **Your turn.** Calculate the dot product with u, the length, the cosine similarity and the Euclidean distance for P2 and P3. Then rank P1, P2 and P3 both ways, as lists of names. Run the check cell. Write your sentence about which measure to use for word counts in `c03-notes.md`.

In [ ]:
p2 = [4, 0, 3]
p3 = [0, 3, 4]

dot_p2, dot_p3 = dot(u, p2), dot(u, p3)
length_p2, length_p3 = length(p2), length(p3)
cosine_p2, cosine_p3 = cosine(u, p2), cosine(u, p3)
distance_p2, distance_p3 = distance(u, p2), distance(u, p3)
candidates = {"P1": p1, "P2": p2, "P3": p3}
rank_cosine = sorted(candidates, key=lambda t: cosine(u, candidates[t]), reverse=True)
rank_distance = sorted(candidates, key=lambda t: distance(u, candidates[t]))

Run the check cell.

In [ ]:
expect("the dot product with P2", dot_p2, 14)
expect("the dot product with P3", dot_p3, 11)
expect("the length of P2", length_p2, 5)
expect("the length of P3", length_p3, 5)
expect("the cosine similarity with P2", cosine_p2, 0.933, tol=0.001)
expect("the cosine similarity with P3", cosine_p3, 0.733, tol=0.001)
expect("the distance to P2", distance_p2, 2.449, tol=0.001)
expect("the distance to P3", distance_p3, 3.464, tol=0.001)
expect("the ranking by cosine similarity", rank_cosine, ["P2", "P1", "P3"])
expect("the ranking by distance", rank_distance, ["P1", "P2", "P3"])

The two measures name a different "most similar" ticket. The reason is length: P2 is longer (5) than u and P1 (3). The model answer and the review rule are on the lesson page, under [Module practice](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/#module-practice).

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you ranked four solved tickets by cosine similarity and by Euclidean distance. T-3907 is first by cosine similarity (1.000) and last by distance (4.472), because cosine similarity ignores length. The cosine similarity of a vector of length 0 is not defined. Counting length in words made length decide 99.96% of the squared distance; standardising with the mean and the SD fixed it. For unit vectors, both measures give the same ranking.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Events and uncertainty](https://learning.nextia-ai.com/courses/math/m03/events-and-uncertainty/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.